# Stage 2 Notebook 46 - Exp2QQ Anchor + Varifocal Loss + IoU regression

**The closed-form fix for the cls collapse.** After 7 experiments (NB39-45) trying every loss tweak, matching change, mask scoring, and capacity scale-up, the cls head was still stuck at pos-neg gap = 0.01 and decoded_f1 = 0.04. I worked out the math on the ASL equilibrium with our actual settings and found the bug: `focal_alpha = 0.25`.

alpha=0.25 is the RetinaNet recommendation for object detection where positives are 1 in 1000. For our 192-anchor lane head, the ratio is only 1:37 -- much less imbalanced. With alpha=0.25, ASL gamma_neg=4, and clip=0.05, the loss has zero gradient at sigmoid ~ 0.578 for ALL priors, which matches exactly what we observed across NB39-45 (val_lane_cls = 0.089, pos_score = 0.578, neg_score = 0.571).

Exp2QQ uses Varifocal Loss (Zhang et al. CVPR 2021, used by RTMDet/VarifocalNet) on the continuous LineIoU regression target:
- Positives: weight = target_iou (no `(1-alpha)` discount; rare positives carry full weight)
- Negatives: weight = alpha * sigmoid^gamma (only confident-wrong negatives count; uniform sigmoid ~ 0.5 gets weight `alpha * 0.25 ~ 0.19`, much smaller than positive's `0.7+`)

This breaks the symmetric equilibrium quantitatively: positives now have ~ 4x the gradient of negatives, so sigmoid is pulled UP for matched priors and DOWN for unmatched ones until they actually separate.

Reference: Zhang et al. 'VarifocalNet: An IoU-aware Dense Object Detector' CVPR 2021. The combination of VFL + continuous IoU target is the published recipe behind RTMDet's COCO-2022 dominance.

Single config diff vs Exp2KK (NB40):
- `cls_target_type: matched_existence -> lineiou_regression`
- `cls_loss_type: asl -> vfl`
- `vfl_alpha: 0.75`, `vfl_gamma: 2.0`
All other settings = NB40 (anchor head, dynamic-k, mask aux, cosine LR, AMP, 20 epochs).

### Run mode

1. `DEBUG_MODE = True` for the first run -- VFL is a new code path; smoke first.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. AMP keeps wall-clock ~ 30 minutes for 20 epochs at 3000 samples.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Independent of NB35-45; only depends on the dataset tar.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp41_rmt_gca_anchor_vfl_iou_regression_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp41_rmt_gca_anchor_vfl_iou_regression_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp41_rmt_gca_anchor_vfl_iou_regression_joint_smoke.log
OK exp41_rmt_gca_anchor_vfl_iou_regression_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.3903 det_loss=3.6872 grad_cos=-0.1688 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49892303347587585, 'gate/lane_mean': 0.5011072754859924, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp41_rmt_gca_anchor_vfl_iou_regression_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp41_rmt_gca_anchor_vfl_iou_regression_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp41_rmt_gca_anchor_vfl_iou_regression_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp41_rmt_gca_anchor_vfl_iou_regression_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp41_rmt_gca_anchor_vfl_iou_regression_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp41_rmt_gca_anchor_vfl_iou_regression_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp41_rmt_gca_anc

0

## What to watch in Exp2QQ training

Reference NB45 (anchor head + ASL + width 1.0 + 30 epochs): pos-neg gap 0.008, cls_only_f1 0.038.
Reference NB40 (anchor head + ASL + width 0.5 + 20 epochs): pos-neg gap 0.010, decoded_f1 0.043.

Pass criteria at epoch 20:
- **`pos_score - neg_score >= 0.10`** -- this is the smoking gun. If VFL fixes the equilibrium, the cls scores should separate by at least 10x the historical gap. If gap >= 0.20, even better.
- **`val/lane/decoded_f1 >= 0.15`** (cls ranking) -- 3x NB40, because cls is now actually discriminative.
- **`val/matched_line_iou >= 0.45`** -- preserves geometry; VFL should not interfere with the geometric losses since it only modifies the cls weighting.
- **`val/lane/decoded_oracle_f1 >= 0.40`** -- oracle ceiling stays high.
- `val_lane_cls` should be HIGHER than NB40's 0.089 in early epochs (because positives now carry full weight and pull harder on the matched priors), then DECREASE as cls actually learns.

Failure signals:
- pos-neg gap still < 0.05: VFL alpha=0.75 too gentle. Try alpha=0.95.
- decoded_f1 < 0.05 with high pos-neg gap: cls is now discriminative but pointing at the wrong priors. Reduce `match_cost_cls` to 0.5 so matching is driven by IoU not cls.
- matched_iou drops below 0.40: VFL is somehow distorting geometry training. Lower `w_cls` to 3.0.

If decoded_f1 jumps to >= 0.15 AND pos-neg gap >= 0.10, **Exp2QQ is the new champion** and we have a working cls signal for the first time.